# Every modality, two questions: modalities as adapters

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 45 §45.2 · type: concept-lab*

*One-line promise:* place **any** modality on a single map — *how does it become context* (image/audio/transcript **in**) and *which tool produces it* (TTS / image-gen **out**) — and watch the agent loop from Part IV stay completely unchanged.

## 🧠 Why this matters

Most of the world's information was never text: scanned invoices, dashboards, whiteboard photos, call recordings. It's tempting to treat "the model can see now" as a new kind of architecture — a multimodal *core*. It isn't. A frontier model takes image and audio blocks the same way it takes text, and producing media is just a **tool call**. So multimodality is a set of **adapters at the edges** of the loop you already built; the loop, memory, and orchestration don't change. Getting that mental model right is what stops you from over-engineering.

## Objectives & prereqs

**By the end you can:**
- Ask the **two questions** of any modality and classify it as *context-in* or *tool-out*.
- Build an **image content block** and an **audio-transcript document loader**, and inspect both shapes.
- Register a *mock* `generate_image` tool and see that an output modality is just another tool.
- Explain why adding vision touches only the **edges** of the Part IV loop.

**Prereqs:** Ch 11 (model APIs, content-block shapes) · Ch 12 (the tool-use loop). Run the setup cell first.

**Cost:** none. Fully offline/mock by design — canned vision and image-gen responses. No key needed.

## Setup

In [ ]:
import base64
import json
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # reads a git-ignored .env if present; never hardcode keys

# MOCK=1 (the default) returns canned, realistic responses so this notebook runs
# FREE, OFFLINE, and DETERMINISTICALLY with no API key. Set COMPANION_MOCK=0 (and
# ANTHROPIC_API_KEY) to hit a live vision/image-gen model once you want real output.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# The book's default model. Never called in MOCK mode; here so the live path is one
# flag away and the code shape matches the book (§45.1).
MODEL = os.getenv("COMPANION_MODEL", "claude-opus-4-8")

DATA = Path("data")

print("MOCK =", MOCK, "| model =", MODEL)
if not MOCK and not os.getenv("ANTHROPIC_API_KEY"):
    raise SystemExit("MOCK=0 needs ANTHROPIC_API_KEY in your environment / .env")

## The whole chapter on one diagram

Hold this picture. The **core** is the Part IV loop you already have; multimodality bolts **adapters** onto its two edges.

```
      INPUT EDGE (context-in)            CORE (unchanged)            OUTPUT EDGE (tool-out)
  image  ─┐                          ┌────────────────────┐                ┌─ generate_image
  audio  ─┼─►  becomes a             │  agent loop +      │  calls a tool  ┤
  pdf    ─┘    content / doc block ─►│  memory + orchestr.│ ──────────────►└─ text_to_speech
  text  ─────►                       └────────────────────┘
```

Two questions decide where anything goes:

1. **How does it become context?** (image block, audio→transcript, pdf→document block) → *input edge.*
2. **Which tool produces it?** (image generation, text-to-speech) → *output edge.*

### Question 1 — context-in: an image content block

A vision input is a `{"type": "image", "source": {...}}` block sitting next to your text block in the same `content` list (§45.1). Nothing else about the request changes. We read the tiny committed `data/sample_page.png` and base64-encode it exactly as you would a real scan.

In [ ]:
img_bytes = (DATA / "sample_page.png").read_bytes()
img_b64 = base64.standard_b64encode(img_bytes).decode()

image_block = {
    "type": "image",
    "source": {"type": "base64", "media_type": "image/png", "data": img_b64},
}
text_block = {"type": "text", "text": "What is on this page? Return one sentence."}

# This is the SAME messages shape as a text-only call — just one extra block.
user_turn = {"role": "user", "content": [image_block, text_block]}

print("image bytes:", len(img_bytes), "| base64 chars:", len(img_b64))
print("content blocks in the turn:", [b["type"] for b in user_turn["content"]])

### Question 1 again — audio as ingestion: a transcript *document loader*

Audio doesn't need a new architecture either. Speech-to-text (the Whisper family and hosted equivalents) turns a recording into **text your existing pipeline already handles** (§45.2). Architecturally it's *just another document loader* — the only new concerns are diarization (who said what) and timestamps. We load a 3-line mock transcript and shape it into the same kind of text block.

In [ ]:
def load_audio_transcript(path):
    """Stand-in for a speech-to-text step: a recording in, plain text out.

    In MOCK we just read a committed transcript; with MOCK=0 you'd call a hosted
    STT model here. Either way the RETURN is text the rest of the agent already eats.
    """
    raw = Path(path).read_text(encoding="utf-8").strip()
    lines = [ln for ln in raw.splitlines() if ln.strip()]
    return {"type": "text", "text": "Meeting transcript:\n" + "\n".join(lines)}


transcript_block = load_audio_transcript(DATA / "meeting_transcript.txt")
print(transcript_block["text"])
print("\n-> note: this is a text block. Audio entered as INGESTION, not a new core.")

### Question 2 — tool-out: register a (mock) `generate_image` tool

Image *generation* is an **output** modality, and the chapter is blunt about what that means: it's not a new pathway, it's **a tool** (§45.2). The agent calls `generate_image(prompt, size, style)`, inspects the result (often with its own vision), and retries or refines — the exact tool-use loop from Ch 12. Here's the tool *schema* and a mock executor; no pixels are actually rendered.

In [ ]:
generate_image_tool = {
    "name": "generate_image",
    "description": "Render an image from a text prompt. Output modality, not an input.",
    "input_schema": {
        "type": "object",
        "properties": {
            "prompt": {"type": "string"},
            "size": {"type": "string", "enum": ["512x512", "1024x1024"]},
            "style": {"type": "string", "enum": ["photo", "diagram", "sketch"]},
        },
        "required": ["prompt"],
    },
}


def generate_image(prompt, size="1024x1024", style="photo"):
    """Mock executor: returns a fake asset handle + provenance, never a real call.

    Keeping PROVENANCE on every generated asset is a senior habit (often a compliance
    requirement, per §45.2's senior lens), so we attach it even in the mock.
    """
    handle = "asset://mock/" + str(abs(hash((prompt, size, style))) % 10_000_000)
    return {
        "url": handle,
        "size": size,
        "style": style,
        "provenance": {"generated": True, "model": MODEL, "prompt": prompt},
    }


print(json.dumps(generate_image_tool, indent=2)[:240], "...")
print("\nmock render:", generate_image("a friendly robot mascot, flat vector", style="diagram"))

### 🔮 Predict

You're about to run one full multimodal turn: an **image goes in**, the agent reasons, and it calls the **`generate_image` tool** to produce an output image. Before you run it —

> **Which parts of the agent loop have to change to support this?** The message-building? The tool-dispatch step? The memory? The orchestration? Write down your guess, then run.

In [ ]:
def run_multimodal_turn(user_content, tools):
    """A deliberately tiny trace of the Part IV loop. The ONLY multimodal-specific
    lines are where blocks are built (input edge) and where a tool is dispatched
    (output edge). The loop control flow is identical to a text-only agent."""
    trace = []

    # 1) Build context (input edge): blocks may be text OR image — loop doesn't care.
    trace.append(("build_context", [b["type"] for b in user_content]))

    # 2) Model reasons. MOCK: it decides to call generate_image. (No network.)
    if MOCK:
        decision = {"stop_reason": "tool_use", "tool": "generate_image",
                    "args": {"prompt": "vector mascot from the sketch", "style": "diagram"}}
    else:  # live path shape — see Ch 12 for the full loop
        from anthropic import Anthropic
        client = Anthropic()
        msg = client.messages.create(model=MODEL, max_tokens=512, tools=tools,
                                      messages=[{"role": "user", "content": user_content}])
        decision = {"stop_reason": msg.stop_reason}
    trace.append(("model_decision", decision["stop_reason"]))

    # 3) Dispatch tool (output edge): generating media is just running a tool.
    if decision.get("tool") == "generate_image":
        result = generate_image(**decision["args"])
        trace.append(("tool_result", result["url"]))

    # 4) (loop would feed the result back; memory/orchestration unchanged)
    return trace


tools = [generate_image_tool]
for step, detail in run_multimodal_turn(user_turn["content"], tools):
    print(f"{step:16} -> {detail}")

**What you just saw.** The trace has four steps. Steps 1 and 3 are the *only* places anything modality-specific happens — building blocks at the input edge, dispatching a tool at the output edge. Step 2 (model reasons) and step 4 (feed back, remember, orchestrate) are byte-for-byte the Part IV loop. The core never learned a new trick.

### ⚠️ Pitfall: mistaking a new input *type* for a new *architecture*

The seductive error is to stand up a separate "vision service," a bespoke "audio agent," and a "media pipeline" — three new cores. The chapter's mental model says no: you added **adapters**, not architectures. Let's prove vision changes only the edge by diffing the two turns.

In [ ]:
text_only_turn = {"role": "user", "content": [text_block]}

def loop_touchpoints(turn):
    # What the loop actually inspects: the list of block TYPES it must build.
    return {
        "block_types": [b["type"] for b in turn["content"]],
        "uses_same_messages_api": True,
        "uses_same_tool_loop": True,
        "uses_same_memory": True,
    }

text_tp = loop_touchpoints(text_only_turn)
mm_tp = loop_touchpoints(user_turn)

print("text-only :", text_tp)
print("multimodal:", mm_tp)
diff = {k for k in text_tp if text_tp[k] != mm_tp[k]}
print("\nwhat changed between them:", diff or "{only the block list at the input edge}")

## 🎯 Senior lens

**Where do you spend the quality budget?** Not evenly. Vision-based *extraction with verification* can replace an entire OCR vendor contract — that's **durable** value, and it's worth real engineering (the whole of `45-02`). Generated images and audio are the opposite: **cheap to produce, expensive to review.** A workflow that assumes a human approves each generated asset scales completely differently from one that publishes autonomously. So: invest at the *input* edge where verification compounds; design an explicit **review boundary** at the *output* edge; and keep **provenance** on every generated asset (increasingly compliance, not just hygiene). Same loop, very different risk profiles at the two edges.

## Recap

- Every modality answers one of **two questions**: *how does it become context* (input edge) or *which tool produces it* (output edge).
- **Vision in** = one extra `image` content block next to your text. **Audio in** = transcribe-then-treat-as-a-document. **Image out** = a `generate_image` *tool*.
- The agent **loop, memory, and orchestration do not change** — multimodality is adapters at the edges, not a new core.
- Spend the quality budget **asymmetrically**: verification at the input edge (durable), a human review boundary + provenance at the output edge.
- Don't mistake a new input *type* for a new *architecture*.

## Exercises

1. **A third input modality.** Add a `load_pdf(path)` loader that returns a `document` block (`media_type: application/pdf`). 🔮 Predict which of the four loop touchpoints, if any, it changes — then show it changes only the block list.
2. **A second output tool.** Define a `text_to_speech(text, voice)` tool schema + mock executor and route a turn through it. Confirm the dispatch step is the only new code.
3. **Classify five modalities.** For webcam frames, a podcast, a chart you must *read*, a chart you must *draw*, and a voice reply — label each *context-in* or *tool-out* and name the adapter.
4. **Provenance audit.** Extend `generate_image`'s provenance with a timestamp and a content hash. Why does the senior lens call this a compliance concern, not just hygiene?

In [ ]:
# Exercise 1 — load_pdf returning a document block; check the touchpoints.

In [ ]:
# Exercise 2 — text_to_speech tool schema + mock executor; route a turn.

In [ ]:
# Exercise 3 — classify five modalities (a dict is fine: name -> (edge, adapter)).

## Next

- **Next notebook:** [`45-02-document-extraction-pipeline.ipynb`](./45-02-document-extraction-pipeline.ipynb) — take the *input edge* seriously: turn a document image into **validated JSON** with a verification gate (the chapter's 🔧 Build).
- **Reuses:** the tool-use loop from [`../../part-04-building-blocks-of-agents/`](../../part-04-building-blocks-of-agents/) (Ch 12) — vision and image-gen are just edges on it.
- **Capstone:** there's no dedicated multimodal module; a `vision`/`extract` tool would plug straight into [`../../../capstone/agents/tools/`](../../../capstone/agents/tools/) as one more adapter on the existing loop.